# 03. RAG 직접 구현 — 연세대학교 대학원 학칙 Q&A
**Day 4**

> 수업용 문서: **`samples/` 안의 연세대학교 대학원 학칙 PDF**

**핵심 데모:**
- RAG **없이** 질문 → 모델이 학칙 세부 조항을 **모름** (환각 가능)
- RAG **있으면** → 학칙 PDF에서 검색한 근거로 **정확히** 답변

---
## 0. 설치

In [ ]:
!pip install openai python-dotenv pymupdf

In [ ]:
import json, os, re, sys
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI

BASE = Path('.').resolve()
sys.path.insert(0, str(BASE))
load_dotenv(BASE.parent / '.env')
client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))

In [ ]:
pdf_path = os.path.join(os.getcwd(),'samples/학칙.pdf')

---
## 1. RAG가 필요한 이유 — 학칙 질문으로 확인


In [ ]:
demo_question = '연세대학교에서 논문제출 시한을 연장한 학생은 휴학할 수 있는가? 예외는 무엇인가?'

plain = client.chat.completions.create(
    model='gpt-4o-mini',
    messages=[{'role': 'user', 'content': demo_question}],
).choices[0].message.content

print('질문:', demo_question)
print('\n=== RAG 없이 (프롬프트만) ===')
print(plain)


---
## 2. Step 1 — 학칙 PDF → 텍스트 추출


In [ ]:
from pdf_summary import pdf_to_text

text_path = pdf_to_text(pdf_path)
with open(text_path, 'r', encoding = 'utf-8') as f:
    rules_text = f.read()
print('학칙 글자 수:', len(rules_text))


In [ ]:
print(rules_text[rules_text.find('제2조의7'):rules_text.find('제2조의7')+350])

---
## 3. Step 2 — 청크(Chunk) 분할

학칙 전문은 약 4만 자 이상 → 검색 가능한 크기로 나눕니다.

In [ ]:
rules_text

In [ ]:
## 앞뒤 빈 줄 공백 제거
## 연속된 공백 문자를 공백 하나로 바꿈
text = re.sub(r'\s+', ' ', rules_text.strip())

In [ ]:
text

In [ ]:
def split_text(text, chunk_size=280, overlap=60):
    text = re.sub(r'\s+', ' ', text.strip())
    if len(text) <= chunk_size:
        return [text]
    chunks, start = [], 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        if end >= len(text):
            break
        start = max(0, end - overlap)
    return chunks

chunks = split_text(rules_text)
print('청크 수:', len(chunks))
print('[C001]', chunks[0][:150], '...')

---
## 4. Step 3 — 인덱스 구축

In [ ]:
INDEX = [
    {
        'chunk_id': f'RULE_C{i:03d}',
        'title': '연세대학교 대학원 학칙',
        'text': chunk,
    }
    for i, chunk in enumerate(chunks, 1)
]
print('인덱스 청크:', len(INDEX))

---
## 5. Step 4 — Retrieval (검색)

수업용: 질문 키워드와 청크 키워드 **겹침 개수**로 순위를 매깁니다.

In [ ]:
def tokenize(text):
    return {t.lower() for t in re.findall(r'[가-힣a-zA-Z0-9]+', text) if len(t) > 1}

def search(query, top_k=3):
    q = tokenize(query)
    scored = []
    for item in INDEX:
        score = len(q & tokenize(item['text']))
        if score > 0:
            scored.append((score, item))
    scored.sort(key=lambda x: x[0], reverse=True)
    return [{'score': s, **it} for s, it in scored[:top_k]]

hits = search(demo_question, top_k=3)
for h in hits:
    print(f"[{h['chunk_id']}] score={h['score']}")
    print(h['text'][:200], '...\n')

---
## 6. Step 5 — Generation (근거 기반 답변)

In [ ]:
demo_question

In [ ]:
def answer_with_rag(question, top_k=3):
    hits = search(question, top_k=top_k)
    if not hits:
        return {'question': question, 'answer': '관련 학칙 조항을 찾지 못했습니다.', 'sources': []}
    context = '\n\n'.join(f"[{h['chunk_id']}] {h['text']}" for h in hits)
    r = client.chat.completions.create(
        model='gpt-4o-mini', temperature=0.1,
        messages=[
            {'role': 'system', 'content': 'Use only provided context from university regulations. Answer in Korean.'},
            {'role': 'user', 'content': f'질문: {question}\n\n학칙 문맥:\n{context}\n\n문맥에 없으면 모른다고 답하고 chunk_id를 bullet로 적으세요.'},
        ],
    )
    return {
        'question': question,
        'answer': r.choices[0].message.content,
        'sources': [h['chunk_id'] for h in hits],
    }

rag_result = answer_with_rag(demo_question)
print('=== RAG 사용 ===')
print(rag_result['answer'])
print('\n출처:', rag_result['sources'])

---
## 7. RAG vs 비-RAG — 학칙 질문 3개 비교

In [ ]:
questions = [
    '연세대학교 대학원 석사학위과정 수업연한은?',
    '박사학위과정 재학연한은 몇 년인가요?',
    '통합과정 재학연한은 몇 년(몇 학기)인가요?',
]

for q in questions:
    plain = client.chat.completions.create(
        model='gpt-4o-mini', messages=[{'role': 'user', 'content': q}]
    ).choices[0].message.content
    rag = answer_with_rag(q)['answer']
    print('\n' + '='*60)
    print('Q:', q)
    print('[RAG 없음]', plain[:180])
    print('[RAG 있음]', rag[:280])

---
## 8. 검색 품질 실험 — chunk_size

청크가 너무 작으면 조항이 잘리고, 너무 크면 검색이 부정확해질 수 있습니다.

In [ ]:
for size in [150, 280, 450]:
  cs = split_text(rules_text, chunk_size=size)
  idx = [{'chunk_id': f'C{i}', 'text': t} for i, t in enumerate(cs, 1)]
  # 임시 search
  q = tokenize('석사학위과정 수업연한')
  best = max((len(q & tokenize(it['text'])), it['chunk_id']) for it in idx)
  print(f'chunk_size={size}, chunks={len(cs)}, best={best}')